In [ ]:
import time, logging
from pathlib import Path
from flofish.experiment import Experiment
from flofish.image import Image
from napari_flofish._reader import read_smfish_json
from glob import glob
import numpy as np
import napari 
from skimage import io

### Read in microscope images

In [ ]:
exp = Experiment.from_cfg_file("E:/flofish/test_s.cerevisiae/config.json")
exp.inputdir = "E:/flofish/test_s.cerevisiae/"
exp.outputdir = "E:/flofish/test_s.cerevisiae/output"
exp.create_image_list()

In [ ]:
for params in exp.images.values():
    logging.info(params)
    my_image = Image.from_dict(params, exp)

    tic = time.time()
    my_image.read_image()
    my_image.read_cells()
    my_image.align()
    my_image.create_grgb()

    # save image (write to dir)
    my_image.save_input_layers()
    my_image.time['01-configure'] = time.time() - tic
    my_image.save_metadata("configure")

### Segment images

In [ ]:
exp.read_image_list_from_jsons()
exp.init_omnipose()

for f in exp.json_files:
    logging.info(f'image: {f}')
    my_image = Image.from_json(f, exp)

    # segment image (~ 02-segment)
    tic = time.time()
    my_image.segment_cells()
    my_image.time['02-segment-cells'] = time.time() - tic

    tic = time.time()
    my_image.segment_dapi()
    my_image.time['02-segment-dapi'] = time.time() - tic

    # postprocess masks
    tic = time.time()
    my_image.postprocess_masks()
    my_image.time['02-segment-pp'] = time.time() - tic

    # indicate in metadata which stage was run last
    my_image.save_metadata("segment")

### Correct cell masks (if necessary)

In [ ]:
# select image to correct cell masks of 
i = 0 # index image
jsons= glob(exp.outputdir+'\*\img.json' )
files = read_smfish_json(jsons[i])
print(fr'selected img {i} of {len(jsons)}')
my_image = Image.from_dict(list(exp.images.values())[i], exp)

In [ ]:
# load files into napari
viewer = napari.Viewer()

viewer.add_image(files[0][0],colormap='grey', visible=True,blending='additive', name='DIC')
viewer.add_labels(files[1][0], visible=True,blending='additive', opacity=.2, name='DIC masks')
viewer.add_labels(files[2][0], visible=False,blending='additive', opacity=.2, name='DIC masks expanded')
viewer.add_image(np.amax(files[3][0],axis=0),colormap='blue',visible=True,blending='additive',opacity=.2, name='DAPI')
viewer.add_labels(files[4][0], visible=True,blending='additive', opacity=.5, name='DAPI masks')

In [ ]:
# Save and overwrite corrected masks.
io.imsave(Path(my_image.savepath) / f'DIC_masks_pp.tif',viewer.layers['DIC masks'].data)
io.imsave(Path(my_image.savepath) / f'DIC_masks_pp_expanded.tif',viewer.layers['DIC masks expanded'].data)
io.imsave(Path(my_image.savepath) / f'DAPI_masks.tif',viewer.layers['DAPI masks'].data)

### Detect spots

In [ ]:
exp.read_image_list_from_jsons()

for f in exp.json_files:
    logging.info(f'image: {f}')
    my_image = Image.from_json(f, exp)

    # detect spots (~ 03-detect-spots)
    tic = time.time()
    my_image.find_focus()
    my_image.filter()
    my_image.detect_spots()
    my_image.time['03-detect-spots'] = time.time() - tic

    # decompose spots (~ 04-decompose-spots)
    tic = time.time()
    my_image.decompose_spots()
    my_image.time['04-decompose-spots'] = time.time() - tic

    logging.info(f'image: {f}')
    
    # assign spots (05-assign-spots)
    my_image.assign_spots()

    # save image (json pickle)
    tic = time.time()
    my_image.save("05")
    my_image.time['05-save'] = time.time() - tic

    my_image.save_metadata("spots")